# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 1
EPOCHS = 100

SEED = 812

In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [ ]:
TOP_K = 10 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = True  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

In [ ]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_cnn1d_flat_v8_1"  # None or "runs/nas_1"

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

In [ ]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Getters

### 4.1. Callbacks

In [ ]:
def get_callbacks(trial: optuna.Trial, backup_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        backup_dir (str): Directory where the backup files will be stored.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )
    
    # Backup and restore the model
    # backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    
    # Model checkpointing
    # checkpoint = callbacks.ModelCheckpoint(
    #     filepath=os.path.join(backup_dir, "checkpoint.h5"),
    #     monitor=monitor,
    #     save_best_only=True,
    #     save_weights_only=True,
    # )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = callbacks.TerminateOnNaN()

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor, interval=3)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, nan_pruner_callback, pruning_callback]


def get_activation(function: str) -> tf.keras.layers.Layer:
    """
    Returns the activation layer based on the provided function name.
    
    Args:
        function (str): Name of the activation function.
        
    Returns:
        tf.keras.layers.Layer: Corresponding activation layer.
    """
    if function == "relu":
        return layers.Activation("relu")
    elif function == "tanh":
        return layers.Activation("tanh")
    elif function == "sigmoid":
        return layers.Activation("sigmoid")
    elif function == "swish":
        return layers.Activation("swish")
    else:
        raise ValueError(f"Unsupported activation function: {function}")

## 5. Hyperparameters

In [ ]:
hparams = HParams(
    activation_choices=[
        #? ReLU family
        #"relu",
        # "leaky_relu",
        # "elu",
        # "celu",
        # "selu",
        #? Smooth ReLU-like and modern variants
        # "softplus",
        # "gelu",
        # "mish",
        "swish",
        # "hard_silu",
        #? Tanh family
        "tanh",
        # "hard_tanh",
        # "softsign",
        #? Sigmoid family
        # "sigmoid",
        # "hard_sigmoid",
        # "log_sigmoid",
        #? Linear and Exponential
        # "linear",
        # "exponential",
        #? Gated and transformer-related
        # "glu",
        # "softmax",
        #? Sparsity and uncommon
        # "hard_shrink",
    ],
    regularizer_choices=[
        "none",
        "l1",
        "l2",
        "l1l2",
    ],
    optimizer_choices=[
        # "SGD",
        # "RMSprop",
        # "Adam",
        # "AdamW",
        # "Adadelta",
        # "Adagrad",
        # "Adamax",
        # "Adafactor",
        # "Nadam",
        # "Ftrl",
        "Lion",
        # "Lamb",
        # "LossScaleOptimizer",
    ],
    scaler_choices=[
        "StandardScaler",
        "MinMaxScaler_0_1",
        "MinMaxScaler_-1_1",
        "RobustScaler",
        "QuantileTransformer",
        "PowerTransformer",
    ],
    l1_value=1e-2,
    l2_value=1e-2,
    min_lr=1e-6,
    max_lr=1e-3,
)

initializer_options = [
    initializers.Zeros(),
    initializers.Ones(),
    initializers.Constant(),
    initializers.RandomNormal(),
    initializers.RandomUniform(),
    initializers.TruncatedNormal(),
    initializers.GlorotNormal(),
    initializers.GlorotUniform(),
    initializers.HeNormal(),
    initializers.HeUniform(),
    initializers.LecunNormal(),
    initializers.LecunUniform(),
    initializers.Identity(),
    initializers.Orthogonal(),
    initializers.VarianceScaling(),
]

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    # Each trial gets a different seed
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————— Data Preprocessing ———————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
        x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

        # Inline one-hot encoding of semantic values
        one_hot_lidar = layers.Lambda(
            lambda x: tf.concat(
                [
                    # “Is there a BS anywhere in the 10 channels?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                    # “Vehicle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                    # “Obstacle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                    # “Free?” → 1 channel (all channels zero)
                    tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
                ],
                axis=-1,
            ),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20, 200, 4),
            name="lidar_transform_to_one_hot",
        )(x_lidar_input)
        # -> (batch, 20, 200, 4)

        # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
        x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(
            one_hot_lidar
        )

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(2,), name="coord_input")

        # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
        x_coord: layers.Layer = layers.Lambda(
            lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20 * 200, 2),
            name="coord_tile_flat",
        )(x_coord_input)

        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
        combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

        # ———————————————————————————————— Conv Layers ——————————————————————————————— #
        conv_layers = 4

        for i in range(conv_layers):
            filters = trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512])

            x = build_cnn1d(
                trial=trial,
                hparams=hparams,
                x=combined if i == 0 else x,
                name_prefix=f"conv1d_{i}",
                # Filters
                filters_range=filters,
                # filters_step=80,
                # Kernel size
                kernel_size_range=(2, 3),
                kernel_size_step=1,
                # Other parameters
                # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            )

            # Add MaxPooling1D after each Conv1D layer
            pool_size = trial.suggest_categorical(f"pool_size_{i}", [2, 3])
            x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

        # ———————————————————————————— Extra dense layers ———————————————————————————— #
        # (batch, length, channels) -> (batch, length * channels)
        x = layers.Flatten(name="flatten")(x)

        x = build_dnn(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="dense_1",
            # Units
            units_range=(200, 300),
            # units_step=50,
            # Dropout
            dropout_rate_range=0.0,
            # dropout_rate_step=0.1,
        )

        x = build_dnn(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix="dense_2",
            # Units
            units_range=(200, 300),
            units_step=10,
            # Dropout
            dropout_rate_range=0.5,
            #dropout_rate_step=0.1,
        )

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax", name="output")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = hparams.get_optimizer(trial)

        model.compile(
            optimizer=optimizer,
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = 64
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, backup_dir),
            verbose=2,
        )

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = compute_params_penalized_loss(loss=loss, model=model, params_penalty_factor=1e-8)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        # ——————————————————————————————— Plot Results ——————————————————————————————— #
        # Configure axis
        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])
        val_loss_best = min(history.history["val_loss"])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ———————————————————————— Save model characteristics ———————————————————————— #
        params = model.count_params()
        trial.set_user_attr("num_params", params)
        bits_per_param = tf.dtypes.as_dtype(POLICY.variable_dtype).size
        trial.set_user_attr("model_size_mb", params * bits_per_param / (1024**2))
        trial.set_user_attr("flops", get_flops(model))
        trial.set_user_attr("macs", get_macs(model))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))

        # ————————————————————————————— Print the results ———————————————————————————— #

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {val_loss_best:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009): {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        print(f"Test loss (s009 full): {test_loss_full:.12f}")
        print(f"Test accuracy (s009 full): {test_acc_full:.4f}\n")

        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ———————————————————————————————————————————————————————————————————————————— #
        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"\n❌ Trial {trial.number} hit OOM (Resource Exhausted)")
        log_exception_to_file(title=f"oom_trial_{trial.number}", error=oom_err, logs_dir=logs_dir)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"\nAn error occurred during the trial execution: {e}")
        log_exception_to_file(title=f"error_trial_{trial.number}", error=e, logs_dir=logs_dir)

        raise  # Re-raise the exception to propagate it
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [ ]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# log_resources(log_dir=resources_dir)

In [ ]:
_monitor_proc = start_monitor(
    pid=os.getpid(),
    log_dir=RUN_DIR,
    custom_title=f"{RUN_DIR}",
    recipients_file="./json/recipients.json",
    credentials_file="./json/credentials.json",
)

## Main

In [ ]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, f"optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "backup": os.path.join(study_dir, "backup"),
        "history": os.path.join(study_dir, "history"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    backup_dir, history_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["backup"],
        dirs["history"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            show_summary=True,
            plot_model=False,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    getter = (lambda t: t.value) if RANK_KEY == "value" else (lambda t: t.user_attrs.get(RANK_KEY, float("nan")))
    top_trials = sorted(
        (t for t in study.trials if (v := getter(t)) is not None and not math.isnan(v)),
        key=getter, reverse=RANK_DESCENDING
    )[:TOP_K]

    for rank, trial in enumerate(top_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_num_params = trial.user_attrs.get("num_params", None)
        trial_model_size = trial.user_attrs.get("model_size_mb", None)
        trial_flops = trial.user_attrs.get("flops", None)
        trial_macs = trial.user_attrs.get("macs", None)

        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)
        trial_test_acc_full = trial.user_attrs.get("test_accuracy_s009_full", None)

        save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            num_params=trial_num_params,
            model_size_mb=trial_model_size,
            flops=trial_flops,
            macs=trial_macs,
            sampler=study.sampler.__class__.__name__,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            test_accuracy_full=trial_test_acc_full,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in top_trials}

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    # Clean up backup and logs directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)

    # ———————————————————————————— Rename Top-K Files ———————————————————————————— #
    for rank, trial in enumerate(top_trials, start=1):
        trial_id = trial.number
        for base_dir, ext in (
            (model_dir, ".keras"),
            (fig_dir, ".png"),
            (history_dir, ".csv"),
        ):
            old_name = f"trial_{trial_id}{ext}"
            old_path = os.path.join(base_dir, old_name)
            if os.path.exists(old_path):
                new_name = f"top_{rank}_{old_name}"
                new_path = os.path.join(base_dir, new_name)
                os.rename(old_path, new_path)

    # ——————————————————————————————— Analyze Study —————————————————————————————— #
    analyze_study(study, fig_dir=fig_dir, table_dir=os.path.join(study_dir, "analysis"))

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(
        1
        for t in study.trials
        if t.state not in {optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED}
    )

    clear_output(wait=True)
    print(f"Training completed with {len(study.trials)} trials.")
    print(f"Number of failed trials: {failed_trials}")

    # ———————————————————————————————————————————————————————————————————————————— #
except Exception as e:
    print(f"\n An error occurred: {e}")
    traceback.print_exc()

    error_log_path = os.path.join(logs_dir, "training_error.log")
    with open(error_log_path, "a") as log_file:
        log_file.write(f"An error occurred during training:\n")
        log_file.write(str(e) + "\n\n")
        log_file.write(traceback.format_exc())
        log_file.write("\n\n\n\n")
finally:
    if NUM_TRIALS > 20: # Skip tests
        notify_training_success(
            recipients_file="./json/recipients.json",
            credentials_file="./json/credentials.json",
            subject=f"🎉 {RUN_DIR} Training Complete",
        )

    stop_monitor(_monitor_proc)